# Bridge repetition-loop diagnostic

Investigates whether the repetition/hallucination cycles seen in `dtw_fixed` bridge
losses are caused by **off-manifold latents** reaching the Whisper decoder.

Key design point: `bridge_inference` runs an **SDE** (sigma_max > 0 injects fresh
Gaussian noise at every reverse step), so each call on the *same* utterance is a
different stochastic draw — some draws may land on-manifold, others may not. This lets
us do a matched comparison: same `z_acc`, same bridge weights, different random outcome
(loop vs. no loop) — and ask whether the looping draws show the
**cross-attention-entropy spike** signature that off-manifold inputs are known to
produce (cf. the padding-region diagnostics — uncorroborated, treat as a hypothesis
to test, not a given).

Reusable for any `bridge_*` model in `results/bridge_eval/` — just change `MODEL_NAME`.


In [39]:
import os
# Jupyter kernels don't inherit `source scripts/slurm_env.sh` -- set the data-dir
# env vars directly so get_split_data_dir() resolves to the real (split-across-volumes)
# paths instead of silently falling back to nonexistent local ones.
os.environ.setdefault('TRAIN_DATA_DIR', '/vol/gpudata/tsv22-train/data/processed/train')
os.environ.setdefault('DEV_DATA_DIR',   '/vol/gpudata/tsv22-dev_test/data/processed/dev')
os.environ.setdefault('TEST_DATA_DIR',  '/vol/gpudata/tsv22-dev_test/data/processed/test')

import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import os, sys
ROOT = '/vol/gpudata/tsv22-fyp/accent-robust-asr'
sys.path.insert(0, ROOT)
os.chdir(ROOT)

from src.experiments.exp2_latent_diffusion_bridge.eval import load_bridge_model, load_encoder_state
from src.experiments.exp2_latent_diffusion_bridge.diffusion import bridge_inference
from src.utils.bridge_utils import get_split_data_dir
from src.utils.model_loader import BASE_MODEL_ID
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.modeling_outputs import BaseModelOutput

ROOT = Path("/vol/gpudata/tsv22-fyp/accent-robust-asr")

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME = 'bridge_dtw_fixed_x0_0.5'   # change to compare other bridge variants
N_STEPS    = 20
SIGMA_MAX  = 0.0                          # must match the value the model was trained/evaluated with
PARAM      = 'x0'                        # 'eps' or 'x0' — read from model config below anyway

# NOTE: eval.py never calls torch.manual_seed -- the only seed(42) in there is
# Python's `random` module, used solely for max_utts_per_speaker subsampling. The
# bridge's SDE noise draws in the cached eval CSV are therefore NOT reproducible
# from a fixed torch seed -- there's no single 'canonical' draw to match. We still
# throw 42 into the mix below (it's the project's de-facto convention seed and as
# good a draw as any), alongside a spread of others for the matched comparison.
SEEDS = [0, 1, 2, 3, 4, 42]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

CKPT = ROOT / f'models/{MODEL_NAME}/checkpoint_best.pt'
cfg  = json.loads((CKPT.parent / 'config.json').read_text())
PARAM = cfg.get('parameterization', PARAM)
# print(f'{MODEL_NAME}: alignment={cfg.get("alignment")}  parameterization={PARAM}  sigma_max(trained)={cfg.get("sigma_max")}')
print(f'{MODEL_NAME}: alignment={cfg.get("alignment")}  parameterization={PARAM}  sigma_max(trained)={SIGMA_MAX}')


device: cuda
bridge_dtw_fixed_x0_0.5: alignment=dtw_fixed  parameterization=x0  sigma_max(trained)=0.0


In [32]:
# ── Load models ───────────────────────────────────────────────────────────────
# NOTE: must use attn_implementation="eager" — sdpa silently drops cross_attentions
# (generate() just returns None for them, no error), which is why output_attentions
# needs an eager backend to actually populate `out.cross_attentions`.
print('Loading bridge + whisper (eager attention so cross-attentions are captured)...')
bridge = load_bridge_model(CKPT, device)
decoder = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, local_files_only=True, attn_implementation="eager"
).to(device).eval()
processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID, local_files_only=True)

mapping = json.loads((ROOT / 'src/experiments/exp2_latent_diffusion_bridge/data/mapping_test.json').read_text())
speech_end_frame = {(d['speaker'], d['utterance_id']): d['speech_end_frame'] for d in mapping}


Loading bridge + whisper (eager attention so cross-attentions are captured)...
[Eval] Baked EMA-averaged weights into model from /vol/gpudata/tsv22-fyp/accent-robust-asr/models/bridge_dtw_fixed_x0_0.5/checkpoint_best.pt


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [40]:
# ── Helpers ───────────────────────────────────────────────────────────────────
MAX_GEN_STEPS = decoder.generation_config.max_length - 4
print(f'MAX_GEN_STEPS = {MAX_GEN_STEPS} (max_length={decoder.generation_config.max_length} - 4 prompt tokens)')

def find_state_path(speaker, uid):
    """Locate the cached [1500, 768] encoder-state .pt for (speaker, utterance_id)."""
    for split in ['train', 'dev', 'test']:
        d = get_split_data_dir(split)
        for fname in [f'{speaker}_{uid}.pt', f'{uid}.pt']:
            cand = d / speaker / fname
            if cand.exists():
                return cand
    return None


def step_entropy(layer_attns):
    """Mean Shannon entropy (nats) of the cross-attention distribution over the
    1500 encoder positions, averaged over layers / heads / query positions, for a
    single decode step. Reference-free: needs only the decoder's own attention —
    no z_nat ground truth required (which the test set lacks).
    Max possible = log(1500) ≈ 7.31 nats (uniform attention -> decoder has no idea
    where to look, cf. the padding-diagnostic 'nat+noise' entropy spike)."""
    vals = []
    for attn in layer_attns:                 # [B, heads, q, kv=1500]
        p = attn.float().clamp_min(1e-12)
        ent = -(p * p.log()).sum(-1)          # [B, heads, q]
        vals.append(ent.mean().item())
    return float(np.mean(vals))


def run_once(z_acc, T_l2, seed):
    """One stochastic bridge-inference + decode pass. Returns (looped, text, per-step entropies).

    `bridge_inference` is an SDE (sigma_max > 0 injects fresh torch.randn noise at
    every reverse step) -- seeding torch's global RNG makes each draw reproducible
    while still letting different seeds explore different stochastic outcomes for
    the *same* utterance. This is what enables the matched loop-vs-no-loop comparison
    below: identical z_acc and bridge weights, the only thing that differs is the
    injected noise."""
    torch.manual_seed(seed)
    with torch.no_grad():
        z_hat = bridge_inference(bridge, z_acc, T_l2=T_l2, T_nat=T_l2,
                                 n_steps=N_STEPS, sigma_max=SIGMA_MAX, parameterization=PARAM).float()
    enc_out = BaseModelOutput(last_hidden_state=z_hat)
    dummy   = torch.zeros(1, 80, 3000, device=device, dtype=z_hat.dtype)
    with torch.no_grad():
        out = decoder.generate(input_features=dummy, encoder_outputs=enc_out,
                               language='en', task='transcribe', temperature=0.0,
                               output_attentions=True, return_dict_in_generate=True)
    text = processor.batch_decode(out.sequences, skip_special_tokens=True)[0]
    ents = [step_entropy(step) for step in out.cross_attentions]
    # A word-count threshold is a weak proxy -- whether a repeated phrase crosses N
    # words after .split() depends on incidental tokenization/punctuation. The robust
    # signature (confirmed in the loss-bucketing analysis) is that genuine loops run
    # all the way to Whisper's max_length ceiling without ever emitting EOS --
    # MAX_GEN_STEPS = generation_config.max_length - 4 prompt tokens
    # (<|startoftranscript|><|en|><|transcribe|><|notimestamps|>) = 448 - 4 = 444,
    # exactly what we observe on every confirmed loop.
    looped = len(ents) >= MAX_GEN_STEPS
    return looped, text, ents


MAX_GEN_STEPS = 444 (max_length=448 - 4 prompt tokens)


In [41]:
# ── Identify the runaway band: utterances with delta > 1.0 vs. baseline ───────
# This is the same cut used in the guardrail transition analysis -- the small
# subset of utterances (typically <1% of the test set) whose WER blew up so badly
# (mean delta ~+20-30) that they account for the bulk of total WER damage. We
# sweep this *whole* band (not a top-N sample) to see what fraction are
# noise-sensitive ("sometimes loops") vs. systematically broken ("always loops").
base = pd.read_csv(ROOT / 'results/whisper_baseline/whisper_baseline.csv').rename(columns={'wer': 'utt_wer'})
df   = pd.read_csv(ROOT / f'results/bridge_eval/{MODEL_NAME}.csv').rename(columns={'wer': 'utt_wer'})

m = base[['utterance_id', 'speaker', 'l1', 'utt_wer']].merge(
        df[['utterance_id', 'speaker', 'utt_wer']],
        on=['utterance_id', 'speaker'], suffixes=('_base', '_bridge'))
m['delta'] = m['utt_wer_bridge'] - m['utt_wer_base']

runaway_band = m[m['delta'] > 1.0].sort_values('delta', ascending=False)
print(f'{len(runaway_band)} utterances in the runaway band (delta > 1.0) for {MODEL_NAME!r}')
print(f'  -> {len(runaway_band) * len(SEEDS)} bridge-inference + decode runs in the sweep below')
runaway_band[['speaker', 'utterance_id', 'l1', 'delta']]


66 utterances in the runaway band (delta > 1.0) for 'bridge_dtw_fixed_x0_0.5'
  -> 396 bridge-inference + decode runs in the sweep below


,speaker,utterance_id,l1,delta
3808,HQTV,arctic_a0541,Vietnamese,62.285714
930,BWC,arctic_b0339,Chinese,58.166667
4185,HQTV,arctic_b0325,Vietnamese,57.800000
1610,EBVS,arctic_a0481,Spanish,54.625000
3619,HQTV,arctic_a0352,Vietnamese,54.375000
...,...,...,...,...
2818,HJK,arctic_b0090,Korean,1.272727
3056,HJK,arctic_b0328,Korean,1.250000
4040,HQTV,arctic_b0180,Vietnamese,1.166667
3530,HQTV,arctic_a0263,Vietnamese,1.142857


In [42]:
# ── Stochastic draw test: does the SAME utterance loop on every draw? ─────────
# If bridge_inference's injected SDE noise is what tips an utterance into a loop,
# re-running it with different seeds should sometimes loop and sometimes not --
# directly testing whether "looping" is a property of the utterance/L1 or of the
# particular noise draw landing off-manifold.
print(f'max possible entropy = log(1500) = {np.log(1500):.2f} nats\n')

draws = []  # (speaker, uid, l1, seed, looped, mean_ent, last10_ent, text)
for i, r in runaway_band.iterrows():
    spk, uid, l1, delta = r['speaker'], r['utterance_id'], r['l1'], r['delta']
    path, T_l2 = find_state_path(spk, uid), speech_end_frame.get((spk, uid))
    if path is None or T_l2 is None:
        print(f'  [skip] missing data for {spk}/{uid}')
        continue

    z_acc = load_encoder_state(path).to(device=device, dtype=torch.bfloat16).unsqueeze(0)
    print(f'--- {spk}/{uid}  (l1={l1}, cached-eval delta={delta:+.2f}) ---')
    for seed in SEEDS:
        looped, text, ents = run_once(z_acc, T_l2, seed)
        tag = 'LOOP  ' if looped else 'normal'
        mean_ent, last10 = float(np.mean(ents)), float(np.mean(ents[-10:]))
        print(f'  {i/runaway_band.shape[0]:.2f}  seed={seed}  {tag}  mean_ent={mean_ent:.3f}  last10_ent={last10:.3f}'
              f'  n_steps={len(ents):4d}  text={text[:70]!r}')
        draws.append((spk, uid, l1, seed, looped, mean_ent, last10, text))

draws_df = pd.DataFrame(draws, columns=['speaker','utterance_id','l1','seed','looped','mean_ent','last10_ent','text'])


max possible entropy = log(1500) = 7.31 nats

--- HQTV/arctic_a0541  (l1=Vietnamese, cached-eval delta=+62.29) ---
  seed=0  normal  mean_ent=4.523  last10_ent=4.523  n_steps=  10  text=' The warring with the war of champion.'
  seed=1  normal  mean_ent=4.523  last10_ent=4.523  n_steps=  10  text=' The warring with the war of champion.'
  seed=2  normal  mean_ent=4.523  last10_ent=4.523  n_steps=  10  text=' The warring with the war of champion.'
  seed=3  normal  mean_ent=4.523  last10_ent=4.523  n_steps=  10  text=' The warring with the war of champion.'
  seed=4  normal  mean_ent=4.523  last10_ent=4.523  n_steps=  10  text=' The warring with the war of champion.'
  seed=42  normal  mean_ent=4.523  last10_ent=4.523  n_steps=  10  text=' The warring with the war of champion.'
--- BWC/arctic_b0339  (l1=Chinese, cached-eval delta=+58.17) ---
  seed=0  normal  mean_ent=4.545  last10_ent=4.522  n_steps=  14  text=" where I'll be plumbed and gossed down."
  seed=1  normal  mean_ent=4.545  

In [36]:
# ── Pooled comparison: do looping draws show elevated cross-attention entropy? ─
# This is the actual test of the off-manifold hypothesis -- reference-free (uses
# only the decoder's own attention behaviour, no z_nat ground truth needed for the
# test-set utterances) and matched (same z_acc / bridge weights; only the
# stochastic outcome differs).
print(f"draws: {len(draws_df)} total — {draws_df['looped'].sum()} looped, {(~draws_df['looped']).sum()} normal\n")

summary = draws_df.groupby('looped')[['mean_ent', 'last10_ent']].agg(['mean', 'std', 'count'])
display(summary)

# Per-utterance: utterances that show BOTH outcomes across seeds are the cleanest
# evidence -- everything held constant except the injected noise.
print('\nUtterances exhibiting both outcomes across seeds (cleanest matched comparison):')
for (spk, uid), g in draws_df.groupby(['speaker', 'utterance_id']):
    if g['looped'].nunique() > 1:
        print(f'  {spk}/{uid}:')
        display(g[['seed', 'looped', 'mean_ent', 'last10_ent']].set_index('seed'))

# ── Bucket each utterance: always loops / sometimes loops / never loops ───────
# This is the headline question -- it tells us whether the dominant failure mode
# is a systematic bad-mean-trajectory problem (fix: bridge training/data/loss) or
# a noise-sensitivity problem (fix: sigma_max / robustness near the manifold edge).
print('\nBucket classification across all swept utterances:')
buckets = draws_df.groupby(['speaker', 'utterance_id', 'l1'])['looped'].agg(['sum', 'count'])
buckets['bucket'] = np.select(
    [buckets['sum'] == 0, buckets['sum'] == buckets['count']],
    ['never_loops', 'always_loops'],
    default='sometimes_loops',
)
display(buckets.reset_index()[['speaker', 'utterance_id', 'l1', 'sum', 'count', 'bucket']])

counts = buckets['bucket'].value_counts()
print(f"\n  always_loops:    {counts.get('always_loops', 0):3d}  (systematic -- fix bridge's mean trajectory)")
print(f"  sometimes_loops: {counts.get('sometimes_loops', 0):3d}  (noise-sensitive -- fix sigma_max / robustness)")
print(f"  never_loops:     {counts.get('never_loops', 0):3d}  (stable at this seed sample size)")


draws: 396 total — 82 looped, 314 normal



mean_ent                 last10_ent                
            mean       std count       mean       std count
looped                                                     
False   4.628327  0.090857   314   4.651803  0.105473   314
True    4.766855  0.049256    82   4.770154  0.056572    82


Utterances exhibiting both outcomes across seeds (cleanest matched comparison):
  BWC/arctic_a0327:


,looped,mean_ent,last10_ent
seed,,,
0,False,4.757921,4.787805
1,False,4.757954,4.792799
2,True,4.766416,4.770438
3,False,4.762120,4.793883
4,False,4.767556,4.805471
42,False,4.770548,4.805210


  BWC/arctic_a0407:


...

  ZHAA/arctic_a0387:


,looped,mean_ent,last10_ent
seed,,,
0,False,4.497797,4.483577
1,True,4.714807,4.727582
2,True,4.709935,4.719028
3,False,4.537312,4.537312
4,False,4.544913,4.544913
42,False,4.462391,4.450493



Bucket classification across all swept utterances:


,speaker,utterance_id,l1,sum,count,bucket
0,BWC,arctic_a0016,Chinese,0,6,never_loops
1,BWC,arctic_a0235,Chinese,0,6,never_loops
2,BWC,arctic_a0315,Chinese,0,6,never_loops
3,BWC,arctic_a0327,Chinese,1,6,sometimes_loops
4,BWC,arctic_a0375,Chinese,0,6,never_loops
...,...,...,...,...,...,...
61,HQTV,arctic_b0453,Vietnamese,2,6,sometimes_loops
62,HQTV,arctic_b0539,Vietnamese,4,6,sometimes_loops
63,ZHAA,arctic_a0269,Arabic,2,6,sometimes_loops
64,ZHAA,arctic_a0387,Arabic,2,6,sometimes_loops



  always_loops:      1  (systematic -- fix bridge's mean trajectory)
  sometimes_loops:  26  (noise-sensitive -- fix sigma_max / robustness)
  never_loops:      39  (stable at this seed sample size)
